# DQN + Precedence Network — CliffWalking-v1

Trains DQN using the World Model (Precedence Network) to **verify actions BEFORE execution**.

✅ Tests all 3 World Model configurations:
- **Config 1** : S(t+1) direct  
- **Config 2** : S(t+2) direct — 2-step lookahead (n=2 chosen over n=5: less compounding error)  
- **Config 3** : ΔS = S(t+1) − S(t) as one-hot delta

✅ Two-part action verification adapted to CliffWalking:
- **Safety check** : predicted next cell ∉ cliff set {37…46}
- **Lookahead check** : k-step greedy simulation from predicted state — detects cliff-adjacent traps

✅ Rich evaluation metrics: reward, cliff rate, goal rate, steps-to-goal, grid heatmaps

## 1. Imports & Setup

In [1]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import gymnasium as gym
from gymnasium.wrappers import TimeLimit
import random
import json
from collections import deque
from datetime import datetime

# ─────────────────────────────────────────────
ENV_NAME        = "CliffWalking-v1"
DATA_PATH       = "../data/collected/CliffWalking"
CHECKPOINT_DIR  = "checkpoints_dqn_precedence_CliffWalking"
PLOT_DIR        = "plots_dqn_precedence_CliffWalking"
WORLD_MODEL_DIR = "world_model/checkpoints_CliffWalking"

# CliffWalking grid constants
GRID_ROWS  = 4
GRID_COLS  = 12
N_STATES   = 48          # 4 × 12
N_ACTIONS  = 4           # up=0, right=1, down=2, left=3
STATE_DIM  = N_STATES    # one-hot dimension
ACTION_DIM = N_ACTIONS
N_STEP_CONFIG2 = 2       # 2-step lookahead (better than 5 for this env)

# Cliff / goal cells
CLIFF_CELLS = set(range(37, 47))   # cells 37..46
GOAL_CELL   = 47
START_CELL  = 36

# DQN hyperparameters
N_EPISODES           = 600
MAX_STEPS_PER_EP     = 200          # same cap as data collector
BATCH_SIZE           = 64
LR                   = 5e-4
GAMMA                = 0.99
EPSILON_START        = 1.0
EPSILON_END          = 0.05
EPSILON_DECAY        = 0.995
TARGET_UPDATE_FREQ   = 10           # episodes between hard copies
BUFFER_SIZE          = 50_000
WARMUP_STEPS         = 500          # transitions before first gradient step

SEED   = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(PLOT_DIR,       exist_ok=True)

np.random.seed(SEED)
torch.manual_seed(SEED)
random.seed(SEED)

print(f"✅ Setup OK")
print(f"  Device      : {DEVICE}")
print(f"  Environment : {ENV_NAME}  ({N_STATES} states, {N_ACTIONS} actions)")
print(f"  Cliff cells : {sorted(CLIFF_CELLS)}")
print(f"  Start={START_CELL} | Goal={GOAL_CELL}")
print(f"  Config 2    : n_step={N_STEP_CONFIG2}")


✅ Setup OK
  Device      : cpu
  Environment : CliffWalking-v1  (48 states, 4 actions)
  Cliff cells : [37, 38, 39, 40, 41, 42, 43, 44, 45, 46]
  Start=36 | Goal=47
  Config 2    : n_step=2


## 2. Environment Helpers

In [2]:
# ── State/grid utilities ─────────────────────────────────────────────────────
def state_to_rc(s):
    """Integer 0-47 → (row, col)."""
    return divmod(s, GRID_COLS)

def one_hot_state(s, n=N_STATES):
    """Integer → one-hot float32 (N_STATES,)."""
    oh = np.zeros(n, dtype=np.float32)
    oh[int(s)] = 1.0
    return oh

def one_hot_action(a, n=N_ACTIONS):
    """Integer → one-hot float32 (N_ACTIONS,)."""
    oh = np.zeros(n, dtype=np.float32)
    oh[int(a)] = 1.0
    return oh

def is_cliff(s):
    return int(s) in CLIFF_CELLS

def is_goal(s):
    return int(s) == GOAL_CELL

def apply_action(s, a):
    """
    Deterministic transition model (CliffWalking physics).
    Returns the theoretical next cell (before Gymnasium's cliff-reset).
    This is used ONLY inside the ActionVerifier lookahead — NOT in training.
    """
    r, c = state_to_rc(int(s))
    if   a == 0: nr, nc = max(0, r-1), c                    # up
    elif a == 1: nr, nc = r, min(GRID_COLS-1, c+1)          # right
    elif a == 2: nr, nc = min(GRID_ROWS-1, r+1), c          # down
    else:        nr, nc = r, max(0, c-1)                     # left
    return nr * GRID_COLS + nc

ACTION_LABELS = ["up(0)", "right(1)", "down(2)", "left(3)"]
print("✅ Environment helpers defined")


✅ Environment helpers defined


## 3. World Model Architecture (load pretrained)

In [9]:
class WorldModel(nn.Module):
    """
    State-prediction network — 3 configurations.

    Config 1 : in = STATE_DIM + ACTION_DIM              → S(t+1) one-hot logits
    Config 2 : in = STATE_DIM + n_step * ACTION_DIM     → S(t+n) one-hot logits
    Config 3 : in = STATE_DIM + ACTION_DIM              → ΔS (S+ΔS in predict())

    All outputs are STATE_DIM-dimensional.
    For action selection we take argmax of the output (treats it as logits / soft one-hot).
    """
    def __init__(self, obs_dim=STATE_DIM, act_dim=ACTION_DIM,
                 config=1, n_step=5, hidden=64):
        super().__init__()
        assert config in (1, 2, 3)
        self.obs_dim = obs_dim
        self.act_dim = act_dim
        self.config  = config
        self.n_step  = n_step if config == 2 else 1

        in_dim = obs_dim + (n_step * act_dim if config == 2 else act_dim)
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, obs_dim),
        )

    def forward(self, state, action_input):
        return self.net(torch.cat([state, action_input], dim=-1))

    def predict(self, state, action_input):
        """Returns absolute next-state prediction (applies S+ΔS for config 3)."""
        out = self.forward(state, action_input)
        return state + out if self.config == 3 else out

    def predict_cell(self, state_oh_np: np.ndarray,
                     action_seq_oh_np: np.ndarray) -> int:
        """
        Convenience: numpy arrays → predicted discrete cell (argmax).
        state_oh_np    : (STATE_DIM,)
        action_seq_oh_np: (ACTION_DIM,) for configs 1&3,
                          (n_step*ACTION_DIM,) for config 2
        """
        s_t = torch.FloatTensor(state_oh_np).unsqueeze(0).to(DEVICE)
        a_t = torch.FloatTensor(action_seq_oh_np).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            out = self.predict(s_t, a_t)
        return int(out.argmax(dim=-1).item())


def load_world_model(config: int, n_step_c2: int = N_STEP_CONFIG2,
                     checkpoint_dir: str = WORLD_MODEL_DIR) -> WorldModel:
    model = WorldModel(STATE_DIM, ACTION_DIM, config=config,
                       n_step=n_step_c2, hidden=64).to(DEVICE)

    # File naming mirrors world_model_CliffWalking.ipynb
    tag = f"config{config}" + ("_n2" if config == 2 else "")
    ckpt_path = os.path.join(checkpoint_dir, f"wm_{ENV_NAME}_{tag}.pt")

    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
        # The architecture only changes the input layer — we load partial weights.
        try:
            model.load_state_dict(ckpt["state_dict"])
            print(f"✅ Config {config} world model loaded: {ckpt_path}")
        except RuntimeError:
            # Input size mismatch (n_step differs) → retrain with n_step=2
            print(f"⚠️  Config {config}: checkpoint n_step mismatch — "
                  f"model needs retraining with n_step={n_step_c2}.")
            print(f"   Using randomly-initialised model for now.")
        model.eval()
        for p in model.parameters():
            p.requires_grad = False
        return model
    else:
        print(f"❌ File not found: {ckpt_path}")
        return None


print("✅ WorldModel architecture defined")


✅ WorldModel architecture defined


## 4. ActionVerifier — CliffWalking Adaptation

### Design rationale

| Check | CartPole | CliffWalking |
|---|---|---|
| **Safety** | pole angle + cart pos < threshold | predicted next cell ∉ cliff set {37…46} |
| **Stability** | Gaussian perturbation variance | **k-step greedy lookahead** from predicted cell |

**Why lookahead instead of perturbation?**  
CliffWalking is fully deterministic and discrete — Gaussian perturbation on a one-hot vector
gives meaningless in-between states. Instead we ask:
*"Even if this step is safe, does it lead to a cliff-adjacent trap within k more steps?"*
This catches the most dangerous pattern: stepping to cell 36→37 (safe-looking left edge of cliff)
which forces a cliff fall on the next step.

**Config 2 action sequence:**  
Build `[a0 | a1]` (n=2):
- `a0` = candidate action  
- `a1` = greedy action from the Q-network on the predicted intermediate state

In [4]:
class ActionVerifier:
    """
    Verifies whether an action is 'good' in CliffWalking by predicting
    the next state with the frozen World Model.

    Strategies
    ----------
    'safety'    : predicted next cell ∉ cliff set {37..46}
    'lookahead' : k-step greedy simulation from predicted cell — rejects
                  actions that lead into cliff-adjacent traps
    'combined'  : safety first (fast), then lookahead if safety passes
    """
    LOOKAHEAD_K = 3          # steps to simulate ahead for trap detection

    def __init__(self, world_model: WorldModel, config: int,
                 aux_model: WorldModel = None,
                 agent_qnet: nn.Module = None):
        """
        Parameters
        ----------
        world_model : main WM (config 1, 2, or 3)
        config      : WM configuration
        aux_model   : Config 1 WM — used to propagate intermediate states
                      in the Config 2 action-sequence rollout
        agent_qnet  : DQN Q-network — greedy actions for Config 2 and lookahead
        """
        self.model      = world_model
        self.config     = config
        self.aux_model  = aux_model
        self.agent_qnet = agent_qnet

        # Verification stats
        self.total_checked   = 0
        self.safety_rejected = 0
        self.lookahead_rejected = 0
        self.no_safe_alt     = 0

    # ── Config 2: build 2-step action vector ─────────────────────────
    def _build_action_seq_config2(self, state_oh: np.ndarray,
                                  first_action: int) -> np.ndarray:
        """
        Builds [a0_oh | a1_oh] for Config 2 (n_step=2).
        a0 = first_action (candidate)
        a1 = greedy action from Q-net on state predicted by aux_model
        """
        n_step = self.model.n_step   # =2
        seq    = []
        s_oh   = state_oh.copy()

        for step in range(n_step):
            a = first_action if step == 0 else self._greedy_action(s_oh)
            seq.append(one_hot_action(a))

            if step < n_step - 1 and self.aux_model is not None:
                # Propagate via Config-1 aux model
                s_t = torch.FloatTensor(s_oh).unsqueeze(0).to(DEVICE)
                a_t = torch.FloatTensor(one_hot_action(a)).unsqueeze(0).to(DEVICE)
                with torch.no_grad():
                    s_oh = self.aux_model.predict(s_t, a_t).cpu().numpy()[0]

        return np.concatenate(seq)    # (n_step * ACTION_DIM,)

    # ── Greedy action from Q-network ─────────────────────────────────
    def _greedy_action(self, state_oh: np.ndarray) -> int:
        if self.agent_qnet is None:
            return random.randint(0, N_ACTIONS - 1)
        s_t = torch.FloatTensor(state_oh).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            return int(self.agent_qnet(s_t).argmax(dim=-1).item())

    # ── Predict next cell using world model ──────────────────────────
    def _predict_next_cell(self, state_oh: np.ndarray, action: int) -> int:
        """Returns predicted next cell as a discrete integer (argmax of WM output)."""
        if self.config == 2:
            a_input = self._build_action_seq_config2(state_oh, action)
        else:
            a_input = one_hot_action(action)
        return self.model.predict_cell(state_oh, a_input)

    # ── k-step lookahead using deterministic env model ───────────────
    def _lookahead_safe(self, start_cell: int) -> bool:
        """
        Simulate k greedy steps from start_cell using apply_action()
        (exact deterministic transitions — no WM error accumulation).
        Returns True if no cliff is hit in k steps.
        """
        cell = start_cell
        for _ in range(self.LOOKAHEAD_K):
            s_oh = one_hot_state(cell)
            a    = self._greedy_action(s_oh)
            cell = apply_action(cell, a)
            if is_cliff(cell):
                return False
        return True

    # ── Main verification ─────────────────────────────────────────────
    def verify(self, state_oh: np.ndarray, action: int,
               strategy: str = 'combined') -> tuple:
        """
        Verify whether 'action' is safe from current state.

        Returns
        -------
        is_valid   : bool
        confidence : float in [0, 1]
        details    : dict with diagnostics
        """
        self.total_checked += 1

        predicted_cell = self._predict_next_cell(state_oh, action)

        details = {
            'strategy'       : strategy,
            'action'         : action,
            'predicted_cell' : predicted_cell,
            'is_cliff'       : is_cliff(predicted_cell),
            'is_goal'        : is_goal(predicted_cell),
            'safety_ok'      : False,
            'lookahead_ok'   : False,
            'is_valid'       : False,
            'confidence'     : 0.0,
        }

        # ── Safety check ─────────────────────────────────────────────
        safety_ok = not is_cliff(predicted_cell)
        details['safety_ok'] = safety_ok

        if strategy in ('safety', 'combined') and not safety_ok:
            self.safety_rejected += 1
            details['is_valid']   = False
            details['confidence'] = 0.0
            return False, 0.0, details

        # ── Lookahead check ───────────────────────────────────────────
        lookahead_ok = True
        if strategy in ('lookahead', 'combined'):
            lookahead_ok = self._lookahead_safe(predicted_cell)
            details['lookahead_ok'] = lookahead_ok

            if not lookahead_ok:
                self.lookahead_rejected += 1
                if strategy == 'lookahead':
                    details['is_valid']   = False
                    details['confidence'] = 0.3    # partial — step is safe but leads to trap
                    return False, 0.3, details

        # ── Final decision ────────────────────────────────────────────
        if strategy == 'safety':
            is_valid   = safety_ok
            confidence = 1.0 if is_valid else 0.0
        elif strategy == 'lookahead':
            is_valid   = lookahead_ok
            confidence = 0.7 if is_valid else 0.3
        else:   # combined
            is_valid   = safety_ok and lookahead_ok
            confidence = (1.0 if safety_ok else 0.0) * (1.0 if lookahead_ok else 0.5)

        details['is_valid']   = bool(is_valid)
        details['confidence'] = float(confidence)
        return is_valid, confidence, details


print("✅ ActionVerifier defined (safety + lookahead strategies)")


✅ ActionVerifier defined (safety + lookahead strategies)


## 5. DQN Agent (one-hot input, compatible with data collector)

In [5]:
class ReplayBuffer:
    """Circular replay buffer storing one-hot states."""
    def __init__(self, capacity=BUFFER_SIZE):
        self.buf = deque(maxlen=capacity)

    def push(self, s_oh, a, r, ns_oh, done):
        self.buf.append((
            np.asarray(s_oh,  dtype=np.float32),
            int(a),
            float(r),
            np.asarray(ns_oh, dtype=np.float32),
            float(done),
        ))

    def sample(self, n):
        batch = random.sample(self.buf, n)
        s, a, r, ns, d = zip(*batch)
        return (
            torch.FloatTensor(np.array(s)).to(DEVICE),
            torch.LongTensor(a).to(DEVICE),
            torch.FloatTensor(r).to(DEVICE),
            torch.FloatTensor(np.array(ns)).to(DEVICE),
            torch.FloatTensor(d).to(DEVICE),
        )

    def __len__(self):
        return len(self.buf)


class QNetwork(nn.Module):
    """
    Q-network for CliffWalking.
    Input : one-hot state (48,)
    Output: Q-values for each action (4,)
    """
    def __init__(self, state_dim=STATE_DIM, n_actions=N_ACTIONS, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.ReLU(),
            nn.Linear(hidden,    hidden), nn.ReLU(),
            nn.Linear(hidden, n_actions),
        )

    def forward(self, x):
        return self.net(x)


print("✅ ReplayBuffer and QNetwork defined")


✅ ReplayBuffer and QNetwork defined


## 6. Training Functions

In [6]:
def dqn_update(q_net, target_net, optimizer, replay):
    """One Double-DQN gradient step. Returns loss or None if buffer not warm."""
    if len(replay) < WARMUP_STEPS:
        return None
    s, a, r, ns, d = replay.sample(BATCH_SIZE)
    q_vals = q_net(s).gather(1, a.unsqueeze(1)).squeeze(1)
    with torch.no_grad():
        best_a = q_net(ns).argmax(1, keepdim=True)
        q_next = target_net(ns).gather(1, best_a).squeeze(1)
        target = r + GAMMA * q_next * (1 - d)
    loss = nn.SmoothL1Loss()(q_vals, target)
    optimizer.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_(q_net.parameters(), 1.0)
    optimizer.step()
    return loss.item()


def train_one_config(config: int,
                     use_precedence: bool = True,
                     verify_strategy: str = 'combined',
                     n_episodes: int = N_EPISODES):
    """
    Train DQN (with or without precedence network) for one WM config.

    Returns a results dict with full training history and evaluation metrics.
    """
    print(f"\n{'='*75}")
    print(f"  DQN — Config {config} | Precedence: {use_precedence} | "
          f"Strategy: {verify_strategy if use_precedence else 'N/A'}")
    print(f"  n_step (Config 2) = {N_STEP_CONFIG2}")
    print(f"{'='*75}\n")

    env = gym.make(ENV_NAME)
    env = TimeLimit(env, max_episode_steps=MAX_STEPS_PER_EP)

    # ── Networks ──────────────────────────────────────────────────────
    q_net      = QNetwork().to(DEVICE)
    target_net = QNetwork().to(DEVICE)
    target_net.load_state_dict(q_net.state_dict())
    target_net.eval()
    optimizer  = optim.Adam(q_net.parameters(), lr=LR)
    replay     = ReplayBuffer()

    # ── World model + verifier ────────────────────────────────────────
    verifier = None
    if use_precedence:
        wm = load_world_model(config)
        if wm is None:
            print(f"❌ Cannot load WM for config {config} — skipping.")
            return None

        aux_model = load_world_model(1) if config == 2 else None

        verifier = ActionVerifier(
            world_model=wm,
            config=config,
            aux_model=aux_model,
            agent_qnet=q_net,     # reference updated in-place as Q-net trains
        )

    # ── Tracking ──────────────────────────────────────────────────────
    ep_rewards, ep_steps, ep_cliff, ep_goal = [], [], [], []
    ep_verified, ep_skipped = [], []
    losses = []
    epsilon = EPSILON_START
    steps_total = 0

    for episode in range(1, n_episodes + 1):
        raw_s, _ = env.reset(seed=SEED + episode)
        s_oh      = one_hot_state(raw_s)
        ep_r      = 0.0
        ep_len    = 0
        fell_cliff = False
        reached_goal = False
        verified_ep = 0
        skipped_ep  = 0

        for _ in range(MAX_STEPS_PER_EP):
            # ── Action selection ──────────────────────────────────────
            if np.random.rand() < epsilon:
                action = env.action_space.sample()
            else:
                with torch.no_grad():
                    s_t = torch.FloatTensor(s_oh).unsqueeze(0).to(DEVICE)
                    action = int(q_net(s_t).argmax(dim=-1).item())

            # ── Verification (if enabled) ─────────────────────────────
            if verifier is not None:
                # Keep Q-net reference fresh for lookahead greedy actions
                verifier.agent_qnet = q_net

                is_valid, confidence, details = verifier.verify(
                    s_oh, action, strategy=verify_strategy)

                if not is_valid:
                    # Try each alternative action
                    found_alt = False
                    for alt in range(N_ACTIONS):
                        if alt == action:
                            continue
                        iv, _, _ = verifier.verify(s_oh, alt, strategy=verify_strategy)
                        if iv:
                            action    = alt
                            found_alt = True
                            break
                    if found_alt:
                        verified_ep += 1
                    else:
                        skipped_ep  += 1   # no safe action found — execute anyway
                        verifier.no_safe_alt += 1
                else:
                    verified_ep += 1

            # ── Env step ──────────────────────────────────────────────
            # Compute theoretical next state BEFORE gym auto-reset on cliff
            theoretical_next = apply_action(raw_s, action)
            raw_ns, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            # Store the theoretical next state (cliff cell) so WM trains correctly
            if reward == -100 and is_cliff(theoretical_next):
                stored_ns = theoretical_next
            else:
                stored_ns = raw_ns

            ns_oh = one_hot_state(stored_ns)

            replay.push(s_oh, action, reward, ns_oh, float(done))
            loss = dqn_update(q_net, target_net, optimizer, replay)
            if loss is not None:
                losses.append(loss)

            ep_r   += reward
            ep_len += 1
            steps_total += 1

            if reward == -100:
                fell_cliff = True
            if stored_ns == GOAL_CELL:
                reached_goal = True

            s_oh   = one_hot_state(raw_ns)   # gym resets cliff → start; use raw_ns
            raw_s  = raw_ns

            if done:
                break

        # ── Episode bookkeeping ───────────────────────────────────────
        ep_rewards.append(ep_r)
        ep_steps.append(ep_len)
        ep_cliff.append(int(fell_cliff))
        ep_goal.append(int(reached_goal))
        ep_verified.append(verified_ep)
        ep_skipped.append(skipped_ep)

        if episode % TARGET_UPDATE_FREQ == 0:
            target_net.load_state_dict(q_net.state_dict())

        epsilon = max(EPSILON_END, epsilon * EPSILON_DECAY)

        if episode % 50 == 0:
            w = min(50, episode)
            avg_r   = np.mean(ep_rewards[-w:])
            avg_clf = np.mean(ep_cliff[-w:])
            avg_gl  = np.mean(ep_goal[-w:])
            print(f"  Ep {episode:4d}/{n_episodes} | "
                  f"AvgR={avg_r:8.2f} | "
                  f"Cliff%={100*avg_clf:.1f} | "
                  f"Goal%={100*avg_gl:.1f} | "
                  f"ε={epsilon:.3f}", end="")
            if use_precedence:
                print(f" | Verified/ep={np.mean(ep_verified[-w:]):.1f} "
                      f"Skipped/ep={np.mean(ep_skipped[-w:]):.1f}")
            else:
                print()

    env.close()

    return {
        'config'          : config,
        'use_precedence'  : use_precedence,
        'verify_strategy' : verify_strategy,
        'q_net'           : q_net,
        'target_net'      : target_net,
        'rewards'         : ep_rewards,
        'steps'           : ep_steps,
        'cliff_flags'     : ep_cliff,
        'goal_flags'      : ep_goal,
        'verified'        : ep_verified,
        'skipped'         : ep_skipped,
        'losses'          : losses,
        'verifier'        : verifier,
    }


print("✅ Training function defined")


✅ Training function defined


## 7. Train — Config 1 (S(t+1) direct)

In [7]:
results_c1 = train_one_config(
    config=1,
    use_precedence=True,
    verify_strategy='combined',
    n_episodes=N_EPISODES
)



  DQN — Config 1 | Precedence: True | Strategy: combined
  n_step (Config 2) = 2

✅ Config 1 world model loaded: world_model/checkpoints_CliffWalking\wm_CliffWalking-v1_config1.pt
  Ep   50/600 | AvgR= -214.52 | Cliff%=18.0 | Goal%=10.0 | ε=0.778 | Verified/ep=192.6 Skipped/ep=2.1
  Ep  100/600 | AvgR= -200.00 | Cliff%=0.0 | Goal%=0.0 | ε=0.606 | Verified/ep=200.0 Skipped/ep=0.0
  Ep  150/600 | AvgR= -201.98 | Cliff%=2.0 | Goal%=0.0 | ε=0.471 | Verified/ep=200.0 Skipped/ep=0.0
  Ep  200/600 | AvgR=  -48.56 | Cliff%=0.0 | Goal%=92.0 | ε=0.367 | Verified/ep=48.5 Skipped/ep=0.1
  Ep  250/600 | AvgR=  -41.64 | Cliff%=0.0 | Goal%=94.0 | ε=0.286 | Verified/ep=41.5 Skipped/ep=0.1
  Ep  300/600 | AvgR=  -40.38 | Cliff%=0.0 | Goal%=94.0 | ε=0.222 | Verified/ep=40.3 Skipped/ep=0.1
  Ep  350/600 | AvgR=  -80.60 | Cliff%=0.0 | Goal%=76.0 | ε=0.173 | Verified/ep=80.5 Skipped/ep=0.1
  Ep  400/600 | AvgR=  -69.10 | Cliff%=0.0 | Goal%=82.0 | ε=0.135 | Verified/ep=68.9 Skipped/ep=0.2
  Ep  450/600 | A

## 8. Train — Config 2 (S(t+2) — 2-step direct)

In [8]:
results_c2 = train_one_config(
    config=2,
    use_precedence=True,
    verify_strategy='combined',
    n_episodes=N_EPISODES
)



  DQN — Config 2 | Precedence: True | Strategy: combined
  n_step (Config 2) = 2

⚠️  Config 2: checkpoint n_step mismatch — model needs retraining with n_step=2.
   Using randomly-initialised model for now.
✅ Config 1 world model loaded: world_model/checkpoints_CliffWalking\wm_CliffWalking-v1_config1.pt
  Ep   50/600 | AvgR=-1539.44 | Cliff%=100.0 | Goal%=4.0 | ε=0.778 | Verified/ep=197.0 Skipped/ep=0.0
  Ep  100/600 | AvgR=-1409.48 | Cliff%=100.0 | Goal%=6.0 | ε=0.606 | Verified/ep=197.7 Skipped/ep=0.0
  Ep  150/600 | AvgR=-1239.06 | Cliff%=96.0 | Goal%=14.0 | ε=0.471 | Verified/ep=189.7 Skipped/ep=0.0
  Ep  200/600 | AvgR= -726.68 | Cliff%=90.0 | Goal%=52.0 | ε=0.367 | Verified/ep=148.5 Skipped/ep=0.0


KeyboardInterrupt: 

## 9. Train — Config 3 (ΔS one-hot delta)

In [ ]:
results_c3 = train_one_config(
    config=3,
    use_precedence=True,
    verify_strategy='combined',
    n_episodes=N_EPISODES
)


## 10. Summary — Training Metrics

In [ ]:
print(f"\n{'='*75}")
print(f"  SUMMARY — DQN + PRECEDENCE NETWORK — {ENV_NAME}")
print(f"{'='*75}\n")

all_results = [r for r in [results_c1, results_c2, results_c3] if r is not None]
WINDOW = 50   # last N episodes for final stats

for res in all_results:
    rw   = res['rewards']
    clf  = res['cliff_flags']
    gl   = res['goal_flags']
    st   = res['steps']
    v    = res['verifier']

    final_r   = np.mean(rw[-WINDOW:])
    final_std = np.std(rw[-WINDOW:])
    cliff_pct = 100 * np.mean(clf[-WINDOW:])
    goal_pct  = 100 * np.mean(gl[-WINDOW:])
    avg_steps = np.mean(st[-WINDOW:])

    # Steps-to-goal (only episodes that reached goal)
    goal_ep_steps = [st[i] for i in range(len(gl)) if gl[i]]
    avg_stg = np.mean(goal_ep_steps[-WINDOW:]) if goal_ep_steps else float('nan')

    print(f"Config {res['config']}  (precedence={res['use_precedence']}):")
    print(f"  Final Mean Reward (last {WINDOW} eps) : {final_r:.2f} ± {final_std:.2f}")
    print(f"  Cliff rate (last {WINDOW} eps)        : {cliff_pct:.1f}%")
    print(f"  Goal  rate (last {WINDOW} eps)        : {goal_pct:.1f}%")
    print(f"  Avg steps/ep (last {WINDOW} eps)      : {avg_steps:.1f}")
    print(f"  Avg steps-to-goal (last {WINDOW} eps) : {avg_stg:.1f}")
    if v is not None:
        total = v.total_checked
        print(f"  WM checks: {total} total | "
              f"safety-rejected={v.safety_rejected} ({100*v.safety_rejected/max(total,1):.1f}%) | "
              f"lookahead-rejected={v.lookahead_rejected} ({100*v.lookahead_rejected/max(total,1):.1f}%) | "
              f"no-safe-alt={v.no_safe_alt}")
    print()

best = max(all_results, key=lambda x: np.mean(x['rewards'][-WINDOW:]))
print(f"🏆 Best config: Config {best['config']}  "
      f"Mean Reward = {np.mean(best['rewards'][-WINDOW:]):.2f}")


## 11. Visualisations

In [ ]:
COLORS = ['#1f77b4', '#ff7f0e', '#2ca02c']
WINDOW_SMOOTH = 25
valid = [r for r in [results_c1, results_c2, results_c3] if r is not None]

fig, axes = plt.subplots(2, 3, figsize=(17, 9))
fig.suptitle(f"DQN + Precedence Network — {ENV_NAME}", fontsize=14)

# ── 1. Learning curves ─────────────────────────────────────────────
ax = axes[0, 0]
for res, col in zip(valid, COLORS):
    rw  = np.array(res['rewards'])
    ep  = np.arange(len(rw))
    ax.plot(ep, rw, alpha=0.15, color=col, linewidth=0.7)
    if len(rw) >= WINDOW_SMOOTH:
        ma = np.convolve(rw, np.ones(WINDOW_SMOOTH)/WINDOW_SMOOTH, mode='valid')
        ax.plot(np.arange(WINDOW_SMOOTH-1, len(rw)), ma,
                color=col, linewidth=2, label=f"Config {res['config']}")
ax.axhline(-13, color='k', linestyle='--', alpha=0.4, linewidth=1, label='Optimal (−13)')
ax.set_xlabel('Episode'); ax.set_ylabel('Return')
ax.set_title('Learning Curves'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

# ── 2. Cliff rate ──────────────────────────────────────────────────
ax = axes[0, 1]
for res, col in zip(valid, COLORS):
    clf = np.array(res['cliff_flags'], dtype=float)
    if len(clf) >= WINDOW_SMOOTH:
        ma = np.convolve(clf, np.ones(WINDOW_SMOOTH)/WINDOW_SMOOTH, mode='valid')
        ax.plot(np.arange(WINDOW_SMOOTH-1, len(clf)), 100*ma,
                color=col, linewidth=2, label=f"Config {res['config']}")
ax.set_xlabel('Episode'); ax.set_ylabel('Cliff rate (%)')
ax.set_title('Cliff Rate (smoothed)'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

# ── 3. Goal rate ───────────────────────────────────────────────────
ax = axes[0, 2]
for res, col in zip(valid, COLORS):
    gl = np.array(res['goal_flags'], dtype=float)
    if len(gl) >= WINDOW_SMOOTH:
        ma = np.convolve(gl, np.ones(WINDOW_SMOOTH)/WINDOW_SMOOTH, mode='valid')
        ax.plot(np.arange(WINDOW_SMOOTH-1, len(gl)), 100*ma,
                color=col, linewidth=2, label=f"Config {res['config']}")
ax.set_xlabel('Episode'); ax.set_ylabel('Goal rate (%)')
ax.set_title('Goal Rate (smoothed)'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

# ── 4. Final performance bar chart ────────────────────────────────
ax = axes[1, 0]
cfg_labels  = [f"Config {r['config']}" for r in valid]
final_means = [np.mean(r['rewards'][-50:]) for r in valid]
final_stds  = [np.std(r['rewards'][-50:])  for r in valid]
bars = ax.bar(cfg_labels, final_means, color=COLORS[:len(valid)],
              yerr=final_stds, capsize=5, alpha=0.85)
for bar, val in zip(bars, final_means):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + (0.5 if val >= 0 else -2.5),
            f"{val:.1f}", ha='center', va='bottom', fontsize=9)
ax.axhline(-13, color='k', linestyle='--', alpha=0.4, linewidth=1, label='Optimal')
ax.set_ylabel('Mean Reward (last 50 eps)')
ax.set_title('Final Performance'); ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)

# ── 5. Verified / Skipped actions ─────────────────────────────────
ax = axes[1, 1]
for res, col in zip(valid, COLORS):
    vf = np.array(res['verified'], dtype=float)
    sk = np.array(res['skipped'],  dtype=float)
    if len(vf) >= WINDOW_SMOOTH:
        ma_v = np.convolve(vf, np.ones(WINDOW_SMOOTH)/WINDOW_SMOOTH, mode='valid')
        ma_s = np.convolve(sk, np.ones(WINDOW_SMOOTH)/WINDOW_SMOOTH, mode='valid')
        x = np.arange(WINDOW_SMOOTH-1, len(vf))
        ax.plot(x, ma_v, color=col, linewidth=1.5,
                label=f"C{res['config']} verified")
        ax.plot(x, ma_s, color=col, linewidth=1.5, linestyle='--',
                label=f"C{res['config']} skipped")
ax.set_xlabel('Episode'); ax.set_ylabel('Actions/episode')
ax.set_title('Verified & Skipped Actions'); ax.legend(fontsize=7); ax.grid(alpha=0.3)

# ── 6. Steps-to-goal distribution ────────────────────────────────
ax = axes[1, 2]
for res, col in zip(valid, COLORS):
    goal_steps = [res['steps'][i] for i in range(len(res['goal_flags']))
                  if res['goal_flags'][i]]
    if goal_steps:
        ax.hist(goal_steps, bins=20, alpha=0.5, color=col,
                label=f"Config {res['config']} (n={len(goal_steps)})")
ax.axvline(13, color='k', linestyle='--', alpha=0.6, label='Optimal (13 steps)')
ax.set_xlabel('Steps to goal')
ax.set_ylabel('Count')
ax.set_title('Steps-to-Goal Distribution (goal episodes only)')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout()
plot_path = os.path.join(PLOT_DIR, 'dqn_precedence_all_configs.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"📊 Plot saved → {plot_path}")


## 12. Grid Heatmaps — Visited States per Config

In [ ]:
def plot_grid_heatmap_agent(q_net, title, save_path, n_eval_eps=200):
    """
    Run n_eval_eps greedy episodes, count state visits, plot heatmap.
    """
    eval_env = gym.make(ENV_NAME)
    eval_env = TimeLimit(eval_env, max_episode_steps=MAX_STEPS_PER_EP)
    visits   = np.zeros(N_STATES, dtype=int)

    q_net.eval()
    for ep in range(n_eval_eps):
        s, _ = eval_env.reset(seed=SEED + 500 + ep)
        done  = False
        while not done:
            visits[s] += 1
            s_oh = one_hot_state(s)
            with torch.no_grad():
                a = int(q_net(torch.FloatTensor(s_oh).unsqueeze(0).to(DEVICE)).argmax().item())
            s, _, term, trunc, _ = eval_env.step(a)
            done = term or trunc
    eval_env.close()
    q_net.train()

    grid = visits.reshape(GRID_ROWS, GRID_COLS)
    fig, ax = plt.subplots(figsize=(12, 4))
    im = ax.imshow(grid, cmap='Blues', aspect='auto')
    plt.colorbar(im, ax=ax, label='Visit count')

    for r in range(GRID_ROWS):
        for c in range(GRID_COLS):
            val = grid[r, c]
            ax.text(c, r, str(val) if val < 10000 else f"{val//1000}k",
                    ha='center', va='center', fontsize=7,
                    color='white' if val > grid.max()*0.6 else 'black')

    # Overlay cliff (red), start (green), goal (gold)
    for c in range(1, 11):
        ax.add_patch(mpatches.Rectangle(
            (c-0.5, 2.5), 1, 1, fill=True, facecolor='tomato', alpha=0.4))
    ax.add_patch(mpatches.Rectangle((-0.5, 2.5), 1, 1,
        fill=False, edgecolor='lime', linewidth=2))
    ax.add_patch(mpatches.Rectangle((10.5, 2.5), 1, 1,
        fill=False, edgecolor='gold', linewidth=2))

    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('Column'); ax.set_ylabel('Row')
    ax.set_xticks(range(GRID_COLS)); ax.set_yticks(range(GRID_ROWS))
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"📊 Heatmap saved → {save_path}")


for res in valid:
    plot_grid_heatmap_agent(
        res['q_net'],
        title=f"Greedy Policy Heatmap — Config {res['config']} (DQN+Precedence)",
        save_path=os.path.join(PLOT_DIR, f"heatmap_config{res['config']}.png"),
        n_eval_eps=200,
    )


## 13. Greedy Evaluation (10 episodes, mean reward)

In [ ]:
def greedy_eval(q_net, n_eval=10, seed_offset=999):
    """Run n_eval greedy episodes. Returns list of episode rewards."""
    eval_env = gym.make(ENV_NAME)
    eval_env = TimeLimit(eval_env, max_episode_steps=MAX_STEPS_PER_EP)
    rewards  = []
    q_net.eval()
    for i in range(n_eval):
        s, _ = eval_env.reset(seed=SEED + seed_offset + i)
        ep_r, done = 0.0, False
        while not done:
            s_oh = one_hot_state(s)
            with torch.no_grad():
                a = int(q_net(
                    torch.FloatTensor(s_oh).unsqueeze(0).to(DEVICE)
                ).argmax().item())
            s, r, term, trunc, _ = eval_env.step(a)
            done  = term or trunc
            ep_r += r
        rewards.append(ep_r)
    eval_env.close()
    q_net.train()
    return rewards


print(f"\n{'='*75}")
print(f"  GREEDY EVALUATION (10 episodes)")
print(f"{'='*75}")
for res in valid:
    ev = greedy_eval(res['q_net'])
    print(f"  Config {res['config']}: "
          f"mean={np.mean(ev):.2f}  std={np.std(ev):.2f}  "
          f"min={min(ev):.0f}  max={max(ev):.0f}  "
          f"goal_rate={100*sum(r > -200 for r in ev)/len(ev):.0f}%")


## 14. Verification Statistics Analysis

In [ ]:
print(f"\n{'='*75}")
print(f"  VERIFICATION STATISTICS")
print(f"{'='*75}\n")

for res in valid:
    v = res['verifier']
    if v is None:
        continue
    config = res['config']
    total  = v.total_checked
    print(f"Config {config}:")
    print(f"  Total WM checks          : {total:,}")
    print(f"  Safety-rejected          : {v.safety_rejected:,}  "
          f"({100*v.safety_rejected/max(total,1):.1f}%)")
    print(f"  Lookahead-rejected       : {v.lookahead_rejected:,}  "
          f"({100*v.lookahead_rejected/max(total,1):.1f}%)")
    print(f"  No safe alternative found: {v.no_safe_alt:,}  "
          f"({100*v.no_safe_alt/max(total,1):.1f}%)")

    # Verification rate over training
    vf = np.array(res['verified'])
    sk = np.array(res['skipped'])
    print(f"  Avg verified actions/ep  : {vf.mean():.2f}")
    print(f"  Avg skipped  actions/ep  : {sk.mean():.2f}")
    print()


## 15. Save Checkpoints & Results

In [ ]:
import json

def save_results(results_list, filename):
    save_data = {}
    for res in results_list:
        if res is None:
            continue
        cfg = res['config']
        rw  = res['rewards']
        gl  = res['goal_flags']
        clf = res['cliff_flags']
        st  = res['steps']

        goal_steps = [st[i] for i in range(len(gl)) if gl[i]]

        save_data[f"config_{cfg}"] = {
            'rewards'             : [float(r) for r in rw],
            'cliff_flags'         : [int(c)   for c in clf],
            'goal_flags'          : [int(g)   for g in gl],
            'steps'               : [int(s)   for s in st],
            'verified'            : [int(v)   for v in res['verified']],
            'skipped'             : [int(s)   for s in res['skipped']],
            # Summary stats
            'final_reward_mean'   : float(np.mean(rw[-50:])),
            'final_reward_std'    : float(np.std(rw[-50:])),
            'final_cliff_pct'     : float(100 * np.mean(clf[-50:])),
            'final_goal_pct'      : float(100 * np.mean(gl[-50:])),
            'avg_steps_to_goal'   : float(np.mean(goal_steps[-50:])) if goal_steps else None,
            'total_goal_episodes' : int(sum(gl)),
            'total_cliff_episodes': int(sum(clf)),
        }
        # Verifier stats
        v = res.get('verifier')
        if v is not None:
            save_data[f"config_{cfg}"]['verifier'] = {
                'total_checked'     : v.total_checked,
                'safety_rejected'   : v.safety_rejected,
                'lookahead_rejected': v.lookahead_rejected,
                'no_safe_alt'       : v.no_safe_alt,
            }

    out_path = os.path.join(CHECKPOINT_DIR, filename)
    with open(out_path, 'w') as f:
        json.dump(save_data, f, indent=4)
    print(f"✅ Results saved → {out_path}")


# Save results JSON
save_results(
    [results_c1, results_c2, results_c3],
    'dqn_precedence_cliffwalking_results.json'
)

# Save model checkpoints
for res in valid:
    cfg      = res['config']
    pth_path = os.path.join(CHECKPOINT_DIR, f"dqn_config{cfg}.pth")
    torch.save({
        'q_net'       : res['q_net'].state_dict(),
        'target_net'  : res['target_net'].state_dict(),
        'config'      : cfg,
        'n_states'    : N_STATES,
        'n_actions'   : N_ACTIONS,
        'use_precedence': res['use_precedence'],
        'verify_strategy': res['verify_strategy'],
    }, pth_path)
    print(f"✅ Model Config {cfg} saved → {pth_path}")

print(f"\n✅ All done!")
print(f"  Checkpoints : {CHECKPOINT_DIR}/")
print(f"  Plots       : {PLOT_DIR}/")
